# SFT + LoRA

## LoRA

**预训练模型在微调过程中，权重更新矩阵 ΔW 具有显著的低秩特性。**
- 直觉上，预训练模型已经学会了丰富的通用表示，微调只需要在这个表示空间中做小幅度的“方向调整”。这种调整展开在一个低维子空间中

前向传播：
$$
W = W_0 + \Delta W
  = W_0 + \frac{\alpha}{r}BA
$$

- $W_0 \in \mathbb{R}^{d \times k}$：冻结的预训练权重，训练期间完全不变，保留模型的通用知识

- $A \in \mathbb{R}^{r \times k}$：下投影矩阵，将 $k$ 维输入压缩到 $r$ 维的低秩空间；使用高斯分布初始化，保证训练开始时有非零梯度

- $B \in \mathbb{R}^{d \times r}$：上投影矩阵，将 $r$ 维低秩表示映射回 $d$ 维输出空间；初始化为全零，确保训练开始时 $\Delta W = BA = 0$，即模型行为与基座模型完全相同，训练稳定性有保障

- $r \ll \min(d, k)$：秩，控制参数量（通常取 8, 16, 32, 64）。$r$ 越小，参数量越少，但表达能力越弱；$r$ 越大，表达能力越强，但参数量增加

- $\frac{\alpha}{r}$：缩放因子，控制 LoRA 更新的幅度。设置为 $\frac{\alpha}{r}$ 而非直接用 $\alpha$，是为了使实际缩放幅度与 $r$ 的选择解耦合：当 $\alpha = 2r$ 时，无论 $r$ 取何值，实际缩放因子始终为 2，方便跨不同 $r$ 值的实验对比


**C = BA 低秩分解的意义：** <br>
<br>
$A \in \mathbb{R}^{r \times k}$，$B \in \mathbb{R}^{d \times r}$<br>
故，$BA \in \mathbb{R}^{d \times k}$，即 $BA$ 的秩为 $r$（因为 $A$ 的秩为 $r$，$B$ 的秩为 $r$）<br>
由于 r ≪ min(d,k)，所以 $BA$ 的秩小于等于 $r$。所以无论如何改变 $A$ 或 $B$ 中的数值（即参数更新），$BA$ 的秩都不会超过 $r$。

**$\alpha$ 与 r 解耦的意义（参数化惯例）：** <br>
<br>
把 LoRA 看作一个团队：
- $r$：团队有多少人
- $BA$：所有人的意见之和
- $\frac{1}{r}$：对意见取平均
- $\alpha$：最终将团队意见放大多少倍

这实际上是一种超参数约定：
- $r$：控制低秩分支的容量；
- $\alpha$：相对于秩设置缩放强度；
- $\alpha / r$：实际乘在 $BA$ 上的缩放值。

## SFT 结束条件

1. 训练/验证 loss 收敛
- 连续 2~3 个 epoch 训练/验证 loss 不再下降（或下降幅度 < 0.01）：结束 SFT
- 训练 loss 持续下降但验证 loss 开始上升：过拟合

2. 格式正确率 >= 90%
- RL 阶段的奖励函数会进一步强化正确格式

3. 任务准确率超过随机水平
- SFT 模型不需要很高的准确率，但需要有意义地超过基座模型的 zero-shot 表现（不经过当前任务的样例训练，直接完成任务的能力）

4. 仍然存在输出多样性
- GRPO 需要采样多个输出，确保模型的多样性

## 建议总结

SFT 只需要让模型学会 Agent 行为的基本格式，需要给 RL 阶段的探索留下足够的空间。

| 建议 | 详情 |
|:---|:---|
| Epoch 数 | 2–3 个 epoch 是安全区间，超过 3 个需要格外警惕过拟合 |
| Early Stopping | 以验证 Loss 为基准，patience=3 eval steps |
| 保存多个检查点 | 保存 epoch 1/2/3 的检查点，RL 阶段可以回退到多样性更好的版本 |
| 先做 RL 预实验 | 用 SFT 检查点做小规模 GRPO（100–200 步），观察 mean_reward 是否上升 |
| 不要追求 SFT SOTA | SFT 的最高准确率 ≠ RL 后的最高准确率 |